# 01 — Data Ingestion & Quality Audit

**Notebook responsibility:** API extraction (Chicago crime + NOAA weather), schema inspection, missingness, duplicates, validation. One responsibility only — cleaning and transformation happen in `02_crime_cleaning` and `05_weather_analysis`.

**Scope:** 2021-01-01 → 2025-12-31, per `src/config.py`.

**Aim:** pull both raw sources, confirm the live schema matches assumptions, and produce a quantified data-quality audit that downstream cleaning decisions will be based on. No rows are modified, dropped, or imputed in this notebook — only measured.

**Output artifact:** `outputs/tables/data_quality.csv` (one row per audit metric, per source).

## 0. Setup

In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.config import (
    START_DATE, END_DATE, NOAA_STATION_ID, CHICAGO_CRIME_DATASET_ID,
    DATA_RAW, OUTPUTS_DIR,
)
from src.data.crime_loader import load_crime_data, save_raw_crime
from src.data.weather_loader import load_weather_data, save_raw_weather
from src.data.validators import validate_crime_data, validate_weather_data

pd.set_option("display.max_columns", 50)
print(f"Scope: {START_DATE} -> {END_DATE}")
print(f"Crime dataset: {CHICAGO_CRIME_DATASET_ID}  |  Weather station: {NOAA_STATION_ID}")

**Findings:** scope confirmed 2021-01-01 → 2025-12-31; dataset ID `ijzp-q8t2`; station `USW00094846`. Config values correct.

## 1. Crime Data Ingestion

**Aim:** pull crime records for the fixed scope via the Socrata API and persist the untouched response to `data/raw/crime/`. Source: City of Chicago Data Portal, *Crimes — 2001 to Present* (`ijzp-q8t2`).

**Rule:** raw data is immutable once saved — no transformation in this cell.

In [ ]:
crime_raw_path = DATA_RAW / "crime" / "crime_raw_2021_2025.parquet"

if crime_raw_path.exists():
    crime_raw = pd.read_parquet(crime_raw_path)
    print(f"Loaded existing raw file: {crime_raw_path}")
else:
    crime_raw = load_crime_data()
    save_raw_crime(crime_raw)
    print(f"Pulled from Socrata API and saved to: {crime_raw_path}")

print(f"Shape: {crime_raw.shape}")

**Findings:** loaded from cache. Shape `(1,210,172, 19)` — 1.21M records over the 5-year window, 19 columns matching `FIELDS`.

### 1.1 Schema inspection

**Aim:** confirm the live API schema matches the fields assumed in `src/data/crime_loader.py`. Do not hard-code assumptions from outdated Kaggle copies of this dataset — the live schema is the source of truth.

In [ ]:
print(crime_raw.dtypes)
print()
crime_raw.head(3)

**Findings:** all 19 expected fields present, none renamed. Socrata's JSON API returns every field as a string except `arrest`/`domestic`, which arrive as native JSON booleans (`bool` dtype) — includes numeric-looking fields (`community_area`, `x_coordinate`, `latitude`, etc.), which is why `validators.py` explicitly `pd.to_numeric(..., errors="coerce")`s them rather than assuming numeric dtype. No dtype surprises beyond this expected pattern.

### 1.2 Schema hard-check (fail fast)

**Aim:** the rest of this pipeline assumes an exact column set. Verify it now and stop immediately if it drifts — a silent schema mismatch discovered three notebooks downstream is far more expensive to debug than one caught here.

In [ ]:
from src.data.crime_loader import FIELDS

missing_fields = set(FIELDS) - set(crime_raw.columns)
extra_fields = set(crime_raw.columns) - set(FIELDS)

if missing_fields:
    raise ValueError(f"Crime schema drift — API no longer returns expected fields: {sorted(missing_fields)}")
if extra_fields:
    print(f"WARNING — API returned unexpected extra fields not in FIELDS: {sorted(extra_fields)}")

print(f"Schema check passed: all {len(FIELDS)} expected fields present.")
print(f"Memory usage: {crime_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")

**Findings:** passed — all 19 fields present, no drift. In-memory footprint 1,192.9 MB (~1.2 GB) for the full 5-year pull; plan for this when later notebooks load the same file (avoid holding multiple full copies simultaneously).

## 2. Weather Data Ingestion

**Aim:** pull daily observations for station `USW00094846` (Chicago O'Hare) for the same 2021–2025 window from NOAA GHCN-Daily, and persist untouched to `data/raw/weather/`.

In [ ]:
weather_raw_path = DATA_RAW / "weather" / "weather_raw_2021_2025.parquet"

if weather_raw_path.exists():
    weather_raw = pd.read_parquet(weather_raw_path)
    print(f"Loaded existing raw file: {weather_raw_path}")
else:
    weather_raw = load_weather_data()
    save_raw_weather(weather_raw)
    print(f"Pulled from NOAA API and saved to: {weather_raw_path}")

print(f"Shape: {weather_raw.shape}")

**Findings:** loaded from cache. Shape `(1826, 7)` — matches the exact expected day count for 2021-01-01→2025-12-31 inclusive (1826 days, accounting for the 2024 leap day).

### 2.1 Schema inspection

**Aim:** confirm TMAX, TMIN, TAVG, PRCP, SNOW, AWND are present, and check the raw dtypes before any numeric comparison is trusted downstream.

In [ ]:
print(weather_raw.dtypes)
print()
weather_raw.head(3)

**Findings:** columns returned are `DATE, AWND, STATION, SNOW, TMAX, TMIN, PRCP` — **`TAVG` is not returned by NOAA for this station/query.** All columns loaded as `object` (string) dtype, including the numeric weather variables — NOAA's Access Data Service JSON, like Socrata, returns quoted strings, not native numbers. Any direct numeric comparison on these columns (e.g. `TMAX < TMIN`) would silently compute a **lexicographic string comparison**, not a numeric one — must coerce before use (handled in 2.2 below).

### 2.2 Column normalization & numeric coercion

**Aim:** standardize `DATE`/`STATION` casing to `date`/`station` so the merge key matches the crime dataset's lowercase convention (required for the Phase 8 merge), and cast the six measurement columns to numeric so comparisons and sentinel checks operate on actual values, not string characters. Operates on the in-memory copy only — `weather_raw` on disk stays untouched.

In [ ]:
weather_raw = weather_raw.rename(columns={c: c.lower() for c in weather_raw.columns if c.upper() in ("DATE", "STATION")})

weather_value_cols = [c for c in ["TMAX", "TMIN", "TAVG", "PRCP", "SNOW", "AWND"] if c in weather_raw.columns]
coercion_failures = {}
for c in weather_value_cols:
    coerced = pd.to_numeric(weather_raw[c], errors="coerce")
    new_nulls = int(coerced.isna().sum() - weather_raw[c].isna().sum())
    if new_nulls > 0:
        coercion_failures[c] = new_nulls
    weather_raw[c] = coerced

print(f"Normalized columns: {list(weather_raw.columns)}")
print(f"Numeric-coerced: {weather_value_cols}")
print(f"Values that failed numeric parsing (became null): {coercion_failures or 'none'}")

_Findings: (fill in after running — confirm 0 coercion failures; if any appear, those values are non-numeric text that slipped into a measurement column and need investigation before Phase 7)_

### 2.3 Schema hard-check (fail fast)

**Aim:** confirm the `date` merge key and the five NOAA variables this station reliably reports are present. `TAVG` is treated as **optional** — 2.1 already established NOAA doesn't return it for this station/query, and the blueprint (Phase 7) already anticipates deriving average temperature from `TMAX`/`TMIN` when needed, so its absence is a known limitation, not a defect to fail on.

In [ ]:
REQUIRED_WEATHER_COLS = {"date", "station", "TMAX", "TMIN", "PRCP", "SNOW", "AWND"}
OPTIONAL_WEATHER_COLS = {"TAVG"}

missing_required = REQUIRED_WEATHER_COLS - set(weather_raw.columns)
missing_optional = OPTIONAL_WEATHER_COLS - set(weather_raw.columns)

if missing_required:
    raise ValueError(f"Weather schema drift — required columns not found: {sorted(missing_required)}")
if missing_optional:
    print(f"NOTE — optional column(s) not returned by this station/query: {sorted(missing_optional)}. "
          f"Derive in Phase 7 as (TMAX + TMIN) / 2 and document as computed, not measured.")

print(f"Schema check passed: 'date' merge key + all {len(REQUIRED_WEATHER_COLS) - 2} required weather variables present.")
print(f"Memory usage: {weather_raw.memory_usage(deep=True).sum() / 1e6:.2f} MB")

**Findings:** schema check passes. `TAVG` is not part of this station's response and is treated as optional, not required. The `date`/`station` merge key is present, independent of the raw file's original column casing, because 2.2 normalizes it on load.

## 3. Crime Data Quality Audit

**Aim:** quantify missingness, duplication, and invalid values using `validate_crime_data()`. This is measurement only — cleaning decisions are made in `02_crime_cleaning`, not here.

In [ ]:
crime_audit = validate_crime_data(crime_raw)

print(f"Row count: {crime_audit['row_count']:,}")
print(f"Duplicate IDs: {crime_audit.get('duplicate_ids')}")
print(f"Duplicate case numbers: {crime_audit.get('duplicate_case_numbers')}")
print(f"Invalid dates: {crime_audit.get('invalid_dates')}")
print(f"Future dates: {crime_audit.get('future_dates')}")
print(f"Invalid coordinates: {crime_audit.get('invalid_coords')}")
print(f"Invalid community areas: {crime_audit.get('invalid_community_area')}")
print()
missing_pct = pd.Series(crime_audit["missing_pct"]).sort_values(ascending=False)
missing_pct[missing_pct > 0]

**Findings:** row count 1,210,172. **0 duplicate IDs** but **155 duplicate case numbers** — these are not the same signal (investigated in 3.1). 0 invalid/future dates. **17,604 invalid coordinates (1.45%)** and **113 invalid community areas (0.009%)** — negligible for the latter, material but small for the former. Missingness above 0%: `longitude`/`latitude`/`x_coordinate`/`y_coordinate` all at 1.45%, `location_description` at 0.52%, `community_area` at 0.009%, `ward` at 0.002%. Nothing exceeds the 5% review threshold.

### 3.1 Extended quality checks (beyond `validate_crime_data`)

**Aim:** deeper checks not covered by the base validator — full-row duplication, coordinate-field completeness, category consistency, near-constant columns, whitespace/casing noise, and boolean-field sanity — since these commonly break downstream feature engineering silently.

In [ ]:
full_row_dupes = int(crime_raw.duplicated().sum())
print(f"Full-row duplicates: {full_row_dupes}")
print("(distinct from duplicate IDs above — a full-row dupe means the entire record repeats, "
      "not just the identifier; a duplicate ID with differing other fields is a re-report/update, not a copy)")

**Findings:** **0 full-row duplicates**, against **155 duplicate case numbers** from Section 3. Since 0 IDs repeat but 155 case numbers do, these are distinct crime records (different `id`, different field values) that share a `case_number` — consistent with one police case generating multiple charge/offense records, not copy-paste duplication. Confirm this interpretation with a manual spot-check on a few repeated case numbers before `02_crime_cleaning` decides whether to keep, flag, or aggregate them.

In [ ]:
for col in ["x_coordinate", "y_coordinate", "latitude", "longitude"]:
    if col in crime_raw.columns:
        null_pct = pd.to_numeric(crime_raw[col], errors="coerce").isna().mean()
        print(f"{col:14s} null/unparseable rate: {null_pct:.2%}")
print("(Chicago's portal blanks coordinates for a small share of records to protect victim privacy "
      "on sensitive offense types — expected, not a defect, but must be handled explicitly in cleaning)")

**Findings:** all four coordinate fields null at exactly **1.45%** — identical to the `invalid_coords` rate in Section 3. This confirms the two checks are measuring the same underlying rows: every "invalid" coordinate is a **blank/null**, not an out-of-bounds value. Single root cause, not two separate issues — `02_crime_cleaning` only needs one missing-coordinate strategy, not a bounds-violation strategy on top.

In [ ]:
category_cols = ["primary_type", "location_description", "iucr", "fbi_code"]
for col in category_cols:
    if col in crime_raw.columns:
        n_raw = crime_raw[col].nunique(dropna=True)
        n_norm = crime_raw[col].dropna().astype(str).str.strip().str.lower().nunique()
        print(f"{col:22s} raw nunique={n_raw:5d}  normalized nunique={n_norm:5d}  "
              f"(diff={n_raw - n_norm} -> whitespace/casing noise if > 0)")

print()
for col in ["arrest", "domestic"]:
    if col in crime_raw.columns:
        print(f"{col} value counts:")
        print(crime_raw[col].value_counts(dropna=False))
        print()

near_constant = {
    c: crime_raw[c].value_counts(normalize=True, dropna=False).iloc[0]
    for c in crime_raw.columns
    if crime_raw[c].nunique(dropna=False) > 0
}
near_constant = {c: p for c, p in near_constant.items() if p > 0.99}
print(f"Near-constant columns (top value > 99% of rows): {list(near_constant.keys()) or 'none'}")

**Findings:** nulls are excluded before casing/whitespace normalization, so `diff` (raw nunique − normalized nunique) reflects only true casing/whitespace collisions and is always ≥ 0.

`primary_type` (31 categories) and `fbi_code` (26) show no casing/whitespace noise. `iucr` has 367 distinct codes — high cardinality, expected for a fine-grained offense code; Phase 4 groups these into the five documented broad categories (Violent/Property/Drug/Fraud/Other), no action needed here. `arrest`: False 1,048,929 / True 161,243 (derived by subtraction from the 1,210,172 total) — clean boolean, no coercion needed. Confirm `location_description` and `domestic` normalized counts, and the near-constant scan result, on execution.

## 4. Weather Data Quality Audit

**Aim:** quantify missingness and date-uniqueness using `validate_weather_data()`, then check for NOAA sentinel and physically implausible values explicitly — the base validator does not catch these.

In [ ]:
weather_audit = validate_weather_data(weather_raw)

print(f"Row count: {weather_audit['row_count']:,}")
print(f"Duplicate dates: {weather_audit.get('duplicate_dates')}")
print()
missing_pct_w = pd.Series(weather_audit["missing_pct"]).sort_values(ascending=False)
missing_pct_w[missing_pct_w > 0]

**Findings:** row count 1,826, **0% missing across every weather column** (confirmed independently via manual `isna().sum()` check). Duplicate-date detection depends on the lowercase `date` column produced by 2.2's normalization step — confirm the reported count is an integer, not `None`, on execution.

### 4.1 Sentinel & plausibility checks

**Aim:** NOAA GHCN-Daily historically uses sentinel codes (e.g. `-9999`) for missing readings in some access paths, and unit mismatches (tenths of a degree/mm vs. standard units) are a common silent error. Confirm neither is present before these values are treated as real measurements.

In [ ]:
weather_vars = ["TMAX", "TMIN", "TAVG", "PRCP", "SNOW", "AWND"]
present_vars = [v for v in weather_vars if v in weather_raw.columns]

sentinel_counts = {v: int((weather_raw[v] == -9999).sum()) for v in present_vars}
print(f"Sentinel (-9999) counts: {sentinel_counts}")
print()

if {"TMAX", "TMIN"}.issubset(weather_raw.columns):
    implausible_temp = int((weather_raw["TMAX"] < weather_raw["TMIN"]).sum())
    print(f"Rows where TMAX < TMIN (physically impossible): {implausible_temp}")

for v in present_vars:
    print(f"{v:6s} min={weather_raw[v].min():>10} max={weather_raw[v].max():>10}  "
          f"(sanity-check against expected units before proceeding)")

**Findings:** these checks require numeric dtype (established at 2.2) — on string-typed columns, `==`, `<`, `.min()`, and `.max()` compare values lexicographically rather than numerically, producing misleading counts and ranges. Confirm the sentinel count, implausible-row count, and min/max range on execution.

## 5. Consolidate & Persist Audit Log

**Aim:** collapse both audits into one flat table and write to `outputs/tables/data_quality.csv`. This file is the single reference downstream notebooks/reviewers use to see what was known about data quality before any cleaning happened.

In [ ]:
def flatten_audit(audit: dict, source: str) -> pd.DataFrame:
    rows = []
    for key, val in audit.items():
        if isinstance(val, dict):
            for sub_key, sub_val in val.items():
                rows.append({"source": source, "metric": f"{key}.{sub_key}", "value": sub_val})
        else:
            rows.append({"source": source, "metric": key, "value": val})
    return pd.DataFrame(rows)

audit_log = pd.concat([
    flatten_audit(crime_audit, "crime"),
    flatten_audit(weather_audit, "weather"),
], ignore_index=True)
audit_log["audited_at_utc"] = datetime.now(timezone.utc).isoformat()

out_path = OUTPUTS_DIR / "tables" / "data_quality.csv"
audit_log.to_csv(out_path, index=False)
print(f"Saved {len(audit_log)} audit rows to: {out_path}")
audit_log.head(10)

**Findings:** 34 audit rows written to `outputs/tables/data_quality.csv` — matches expectation (crime audit metrics + weather audit metrics, with weather short by one entry versus a 6-variable assumption since `TAVG` is absent). File confirmed written; row count as expected.

## 6. Summary

**Top findings:**
- Crime: 1,210,172 records / 19 fields, 2021–2025, schema exactly as expected.
- Weather: 1,826 daily records / 7 fields — **`TAVG` is not available** for station `USW00094846` via this API path; the other five variables (TMAX, TMIN, PRCP, SNOW, AWND) are fully populated with 0% missingness.
- Coordinate/location nulls (1.45% of crime rows) are a single root cause — privacy-masked blanks — not out-of-bounds errors; one missing-coordinate strategy covers both.
- 155 duplicate case numbers with 0 full-row duplicates and 0 duplicate IDs — legitimate multi-record cases, not copy-paste duplication (pending a manual spot-check to confirm).
- Both raw sources arrive from their APIs as strings for every numeric-looking field — this is expected Socrata/NOAA JSON behavior, not a defect, but every downstream numeric operation must coerce explicitly rather than trust the loaded dtype.

**Quality issues to address in `02_crime_cleaning` / `05_weather_analysis`:**
- Decide the missing-coordinate strategy for the 1.45% of crime rows with blank location data (drop / keep as missing / impute from `block`).
- Confirm the 155 duplicate-case-number records are legitimate (multi-offense incidents) before any per-case aggregation logic is written.
- 113 invalid `community_area` rows (0.009%) — decide drop vs. recover from lat/long before spatial aggregation.
- Map `iucr`'s 367 codes to the five documented broad categories (Violent/Property/Drug/Fraud/Other) per the blueprint's Phase 4 rule against undocumented mappings.
- No `TAVG` from NOAA — derive `avg_temp = (TMAX + TMIN) / 2` in Phase 7 and document it explicitly as computed, not measured, per the blueprint's ban on undocumented composites.
- Execute 2.2 and 4.1 to obtain the weather sentinel count, implausible-row count, and coercion-failure count — these require numeric dtype, established at 2.2.

**Open questions / hypotheses:**
- Are the 155 duplicate case numbers concentrated in specific offense types (e.g. incidents that legitimately produce multiple charges) or spread randomly — would indicate a portal artifact instead?
- Does the 1.45% missing-coordinate rate concentrate in particular years, districts, or offense types (a systematic reporting gap) or is it uniformly distributed (random omission)?

**Next step:** proceed to `02_crime_cleaning.ipynb` — resolve the issues logged above; do not clean in this notebook.